## Azure ML setup
Run this on an Azure ML compute instance with the **Python 3.10 - SDK v2** kernel. The raw data is read from the `athletes_raw` data asset (version 1) in the Data tab. The training jobs run on serverless compute.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import glob, json, os, time
import numpy as np
import pandas as pd
import mltable
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.store.artifact.artifact_repository_registry import get_artifact_repository
from azure.ai.ml import MLClient, Input, automl
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

TARGET = "deadlift"

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)

Found the config file in: /config.json


## Data prep
Load the raw data from the `athletes_raw` data asset, keep the useful columns, and remove missing and impossible values.

In [2]:
data_asset = ml_client.data.get("athletes_raw", version="1")
df = mltable.load(f"azureml:/{data_asset.id}").to_pandas_dataframe()

# remove rows with missing key values, then columns that are not relevant
df = df.dropna(subset=["region", "age", "weight", "height", "howlong", "gender", "eat",
                       "train", "background", "experience", "schedule",
                       "deadlift", "candj", "snatch", "backsq"])
df = df.drop(columns=["affiliate", "team", "name", "athlete_id", "fran", "helen", "grace",
                      "filthy50", "fgonebad", "run400", "run5k", "pullups", "train"])

# remove outliers
df = df[df["weight"] < 1500]
df = df[df["gender"] != "--"]
df = df[df["age"] >= 18]
df = df[(df["height"] < 96) & (df["height"] > 48)]
df = df[((df["deadlift"] > 0) & (df["deadlift"] <= 1105))
        | ((df["gender"] == "Female") & (df["deadlift"] <= 636))]
df = df[(df["candj"] > 0) & (df["candj"] <= 395)]
df = df[(df["snatch"] > 0) & (df["snatch"] <= 496)]
df = df[(df["backsq"] > 0) & (df["backsq"] <= 1069)]

# clean survey data
df = df.replace({"Decline to answer|": np.nan})
df = df.dropna(subset=["background", "experience", "schedule", "howlong", "eat"])

df.shape

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to i

(30029, 14)

## Helper functions
`train` starts an Automated ML regression job on serverless compute and waits for it. The settings are the same as in the Studio run: 5-fold cross-validation, a 30% test split, a 15 minute experiment timeout and all algorithms allowed. It also returns the total time of the job. `leaderboard` lists the finished trials by RMSE with their training time.

In [3]:
def make_table(name, columns):
    folder = f"tables/{name}"
    os.makedirs(folder, exist_ok=True)
    df[columns + [TARGET]].to_csv(f"{folder}/data.csv", index=False)
    lines = ["type: mltable", "paths:", "  - file: ./data.csv", "transformations:",
             "  - read_delimited:", "      delimiter: ','",
             "      header: all_files_same_headers", "      encoding: utf8"]
    with open(f"{folder}/MLTable", "w") as f:
        f.write("\n".join(lines) + "\n")
    return Input(type=AssetTypes.MLTABLE, path=folder)


def train(name, columns):
    # same settings as the Studio run: 5-fold cross-validation, 30% test split,
    # all algorithms allowed. Both jobs are capped at 20 trials.
    job = automl.regression(
        experiment_name=name,
        training_data=make_table(name, columns),
        target_column_name=TARGET,
        primary_metric="normalized_root_mean_squared_error",
        n_cross_validations=5,
        test_data_size=0.3,
    )
    # the per-trial timeout must be below the experiment timeout
    job.set_limits(timeout_minutes=60, trial_timeout_minutes=14, max_trials=20,
                   max_concurrent_trials=4, enable_early_termination=False)
    start = time.time()
    submitted = ml_client.jobs.create_or_update(job)
    print(submitted.studio_url)
    ml_client.jobs.stream(submitted.name)
    total_secs = time.time() - start  # total wall-clock time from submit to finish
    print(f"Total job time: {total_secs / 60:.1f} min")
    return submitted.name, total_secs


def leaderboard(name):
    runs = mlflow.search_runs(experiment_names=[name])
    runs = runs[runs["tags.mlflow.parentRunId"].notna() & runs["metrics.root_mean_squared_error"].notna()]
    out = pd.DataFrame({
        "run_id": runs["run_id"],
        "algorithm": runs.get("tags.run_algorithm", runs["tags.mlflow.runName"]),
        "rmse": runs["metrics.root_mean_squared_error"],
        "train_secs": (runs["end_time"] - runs["start_time"]).dt.total_seconds(),
    })
    return out.sort_values("rmse").reset_index(drop=True)

## All features

In [4]:
all_cols = [c for c in df.columns if c != TARGET]

all_job, all_secs = train("athletes-all", all_cols)
board = leaderboard("athletes-all")
board

https://ml.azure.com/runs/neat_energy_tb9tnsb036?wsid=/subscriptions/138cf905-0a8e-4c60-9ad5-c006f63536bc/resourcegroups/ml/workspaces/mlops&tid=83b02c92-5f26-48ed-9e5b-6c2fca46a8e6
RunId: neat_energy_tb9tnsb036
Web View: https://ml.azure.com/runs/neat_energy_tb9tnsb036?wsid=/subscriptions/138cf905-0a8e-4c60-9ad5-c006f63536bc/resourcegroups/ml/workspaces/mlops

Execution Summary
RunId: neat_energy_tb9tnsb036
Web View: https://ml.azure.com/runs/neat_energy_tb9tnsb036?wsid=/subscriptions/138cf905-0a8e-4c60-9ad5-c006f63536bc/resourcegroups/ml/workspaces/mlops

Total job time: 41.3 min


,run_id,algorithm,rmse,train_secs
0,jolly_camel_m6n2gt6mqs_20,gray_steelpan_sjn7cp32,36.175797,56.151
1,neat_energy_tb9tnsb036_18,maroon_bee_2rd2ysjl,36.177462,55.950
2,neat_energy_tb9tnsb036_19,yellow_sail_0ncbvm3y,36.204879,59.984
3,jolly_camel_m6n2gt6mqs_21,keen_farm_6tkjdm4n,36.206950,57.692
4,neat_energy_tb9tnsb036_0,calm_fly_q08q7rr1,36.394839,34.182
...,...,...,...,...
81,neat_energy_tb9tnsb036_16,sweet_cart_q9r81725,39.450945,32.198
82,neat_energy_tb9tnsb036_10,cool_wire_tp6cymhv,40.075848,36.674
83,jolly_camel_m6n2gt6mqs_10,olive_oven_x2w84jnj,40.394075,32.374
84,mango_double_35p72gf3z2_9,lime_spider_0g8z517v,41.720298,21.454


## Top 5 features
Azure ML saves a feature explanation for the best model. The importance scores are read from the job's files.

In [6]:
client = MlflowClient()

# trials of the new all-features job only, best first
new_trials = leaderboard("athletes-all")
new_trials = new_trials[new_trials.run_id.str.startswith(all_job + "_")].reset_index(drop=True)

best_run = None
for run_id in new_trials.run_id:
    if any(a.path == "explanation" for a in client.list_artifacts(run_id)):
        best_run = run_id
        break
print("Explanation taken from:", best_run)

os.makedirs("exp", exist_ok=True)
repo = get_artifact_repository(client.get_run(best_run).info.artifact_uri)
repo.download_artifacts("explanation", dst_path="exp")

# the explanation that matches the original columns (not the one-hot encoded ones)
for folder in glob.glob("exp/explanation/*"):
    names = json.load(open(f"{folder}/global_names/0.interpret.json"))
    values = json.load(open(f"{folder}/global_values/0.interpret.json"))
    if set(names) <= set(all_cols):
        ranked = pd.Series(values, index=names).sort_values(ascending=False)

ranked.head(5)


Explanation taken from: neat_energy_tb9tnsb036_18


backsq    51.963497
candj     17.173673
gender     5.648659
weight     4.301335
snatch     2.568607
dtype: float64

## Top 3 models
By validation score (lowest RMSE on the 20% validation split) and by speed (shortest training time).

In [7]:
print("Top 3 by validation score")
print(board.head(3))
print()
print("Top 3 by speed (training time)")
board.sort_values("train_secs").head(3)

Top 3 by validation score
                      run_id               algorithm       rmse  train_secs
0  jolly_camel_m6n2gt6mqs_20  gray_steelpan_sjn7cp32  36.175797      56.151
1  neat_energy_tb9tnsb036_18     maroon_bee_2rd2ysjl  36.177462      55.950
2  neat_energy_tb9tnsb036_19    yellow_sail_0ncbvm3y  36.204879      59.984

Top 3 by speed (training time)


,run_id,algorithm,rmse,train_secs
60,great_chaconia_rmljg84048_6,modest_cup_4x4m8fmy,38.548493,20.940
73,great_chaconia_rmljg84048_1,kind_helmet_cdgbl8df,38.600142,21.018
84,mango_double_35p72gf3z2_9,lime_spider_0g8z517v,41.720298,21.454


## Top 3 features only

In [8]:
top3 = ranked.index[:3].tolist()
print(top3)

top3_job, top3_secs = train("athletes-top3", top3)
board3 = leaderboard("athletes-top3")

print("Top 3 by validation score")
print(board3.head(3))
print()
print("Top 3 by speed (training time)")
board3.sort_values("train_secs").head(3)

['backsq', 'candj', 'gender']
https://ml.azure.com/runs/green_collar_29y0mszt1n?wsid=/subscriptions/138cf905-0a8e-4c60-9ad5-c006f63536bc/resourcegroups/ml/workspaces/mlops&tid=83b02c92-5f26-48ed-9e5b-6c2fca46a8e6
RunId: green_collar_29y0mszt1n
Web View: https://ml.azure.com/runs/green_collar_29y0mszt1n?wsid=/subscriptions/138cf905-0a8e-4c60-9ad5-c006f63536bc/resourcegroups/ml/workspaces/mlops

Execution Summary
RunId: green_collar_29y0mszt1n
Web View: https://ml.azure.com/runs/green_collar_29y0mszt1n?wsid=/subscriptions/138cf905-0a8e-4c60-9ad5-c006f63536bc/resourcegroups/ml/workspaces/mlops

Total job time: 23.3 min
Top 3 by validation score
                       run_id               algorithm       rmse  train_secs
0  green_collar_29y0mszt1n_18  crimson_eagle_m31kxswq  36.981031      52.937
1    olive_comb_2vdtx7tx1m_21   amusing_gold_qt0371cb  36.982141      56.609
2    olive_comb_2vdtx7tx1m_22    gentle_sand_5696mlj1  37.010141      61.233

Top 3 by speed (training time)


,run_id,algorithm,rmse,train_secs
38,zen_drawer_t88sldggs2_5,plucky_ice_bczhqkbk,39.108959,22.707
39,zen_drawer_t88sldggs2_3,maroon_ice_453w8z4k,39.109403,23.013
50,zen_drawer_t88sldggs2_1,nifty_vase_qs946zzb,39.790212,23.318


## Comparison with Part A
Fill in the Part A validation RMSE and prediction time per row. Azure ML does not report prediction time per row for the AutoML models, so its speed is the training time above.

In [9]:
pd.DataFrame(
    [["All features (Part A)", board.rmse.iloc[0], board.train_secs.iloc[0], all_secs],
     ["Top 3 features", board3.rmse.iloc[0], board3.train_secs.iloc[0], top3_secs]],
    columns=["model", "rmse", "best_model_train_secs", "total_job_secs"],
)

,model,rmse,best_model_train_secs,total_job_secs
0,All features (Part A),36.175797,56.151,2480.975621
1,Top 3 features,36.981031,52.937,1398.095917


## Is Azure ML Automated ML no-code, low-code or full-code?
**No-code in Azure ML Studio, low-code in this notebook.** In Studio you can pick the data, the target column and the limits in a form and start the run without writing code. Here the same steps are one `automl.regression(...)` call plus a few settings (trial limit, time limit, allowed algorithms), and Azure ML chooses the preprocessing, models and tuning itself.